## 1 - Tự tính khoảng cách và tìm láng giềng

In [1]:
# -*- coding: utf-8 -*-


import numpy as np
import pandas as pd

# Ba bạn: A và C học gần như y hệt nhau, B học khác hẳn
ba_ban = {
    "A": np.array([5.0, 3.0, 4000.0]),
    "B": np.array([25.0, 9.0, 4100.0]),
    "C": np.array([6.0, 3.5, 7000.0]),
}


def kc(u, v):
    """Khoang cach Euclid giua hai diem."""
    return np.sqrt(((u - v) ** 2).sum())


print("Khoang cach KHI CHUA chuan hoa")
print(f"d(A, B) = {kc(ba_ban['A'], ba_ban['B']):8.1f}")
print(f"d(A, C) = {kc(ba_ban['A'], ba_ban['C']):8.1f}")
print()

# Chuẩn hoá bằng trung bình và độ lệch chuẩn của cảbộ dữ liệu
df = pd.read_csv("data/sinh_vien_mo_rong.csv")
X = df[["gio_on", "diem_giua_ky", "chi_tieu"]].to_numpy(dtype=float)
tb = X.mean(axis=0)
dl = X.std(axis=0)
print("Do lech chuan cua ba cot:")
print(f"gio_on {dl[0]:.2f}    diem_giua_ky {dl[1]:.2f}"
      f"    chi_tieu {dl[2]:.0f}")
print()

chuan = {ten: (v - tb) / dl for ten, v in ba_ban.items()}
print("Khoang cach SAU KHI chuan hoa")
print(f"d(A, B) = {kc(chuan['A'], chuan['B']):.2f}")
print(f"d(A, C) = {kc(chuan['A'], chuan['C']):.2f}")
print()

# Tìm 5 láng giềng gần nhất của một bạn mới, chỉdùng hai cột học tập
X2 = df[["gio_on", "diem_giua_ky"]].to_numpy(dtype=float)
y = df["qua_mon"].to_numpy()
tb2, dl2 = X2.mean(axis=0), X2.std(axis=0)
ban_moi = np.array([12.0, 6.5])
X2c = (X2 - tb2) / dl2
moi_c = (ban_moi - tb2) / dl2
khoang = np.sqrt(((X2c - moi_c) ** 2).sum(axis=1))
gan = np.argsort(khoang)[:5]

print("Ban moi: on 12 gio, diem giua ky 6.5")
print("Nam lang gieng gan nhat:")
for i in gan:
    ket = "qua" if y[i] == 1 else "rot"
    print(f"gio {X2[i, 0]:5.1f}  diem {X2[i, 1]:4.1f}"
          f"  khoang cach {khoang[i]:.3f} -> {ket}")
so_qua = int(y[gan].sum())
print(f"Bo phieu: {so_qua} qua, {5 - so_qua} rot")
print("Du doan:", "qua mon" if so_qua >= 3 else "rot mon")


Khoang cach KHI CHUA chuan hoa
d(A, B) =    102.2
d(A, C) =   3000.0

Do lech chuan cua ba cot:
gio_on 7.82    diem_giua_ky 2.09    chi_tieu 1692

Khoang cach SAU KHI chuan hoa
d(A, B) = 3.85
d(A, C) = 1.79

Ban moi: on 12 gio, diem giua ky 6.5
Nam lang gieng gan nhat:
gio  10.5  diem  6.4  khoang cach 0.198 -> qua
gio  12.5  diem  6.1  khoang cach 0.202 -> qua
gio  14.0  diem  6.4  khoang cach 0.260 -> qua
gio   9.5  diem  6.1  khoang cach 0.373 -> rot
gio  10.5  diem  5.8  khoang cach 0.386 -> qua
Bo phieu: 4 qua, 1 rot
Du doan: qua mon


## 2 - Chuẩn hóa dữ liệu bằng StandardScaler

In [2]:
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/sinh_vien_mo_rong.csv")
y = df["qua_mon"]

X3 = df[["gio_on", "diem_giua_ky", "chi_tieu"]]
X_tr, X_te, y_tr, y_te = train_test_split(
    X3, y, test_size=0.25, random_state=17, stratify=y
)


def cham(mo_hinh, cot):
    mo_hinh.fit(X_tr[cot], y_tr)
    return accuracy_score(y_te, mo_hinh.predict(X_te[cot]))


def knn_chuan():
    """KNN co chuan hoa, gop lam mot khoi bang pipeline."""
    return make_pipeline(StandardScaler(), KNeighborsClassifier(7))


ba_cot = ["gio_on", "diem_giua_ky", "chi_tieu"]
hai_cot = ["gio_on", "diem_giua_ky"]

a = cham(KNeighborsClassifier(n_neighbors=7), ba_cot)
b = cham(knn_chuan(), ba_cot)
c = cham(knn_chuan(), hai_cot)

print(f"Doan bua ca lop qua: {y_te.mean():.4f}")
print(f"Ba cot, khong chuan hoa: {a:.4f}")
print(f"Ba cot, co chuan hoa: {b:.4f}")
print(f"Bo cot chi_tieu, co chuan hoa: {c:.4f}")


Doan bua ca lop qua: 0.6000
Ba cot, khong chuan hoa: 0.5333
Ba cot, co chuan hoa: 0.8667
Bo cot chi_tieu, co chuan hoa: 0.9000


## 3 - Chọn k bằng Cross-Validation

In [3]:
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai cột x1 và x2 có thang đo gần nhau, độ lệch chuẩn chỉchênh khoảng
# 1.6 lần, nên ởbộnày không chuẩn hoá cũng được.
print("Accuracy tren tap hoc va tap kiem tra theo k")
print("  k      tap hoc    tap kiem tra")
for k in [1, 5, 13, 31, 101]:
    mh = KNeighborsClassifier(n_neighbors=k).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{k:3d}      {a_hoc:.4f}        {a_kt:.4f}")
print()

# Chọn k chỉdựa trên tập học, không nhìn tập kiểm tra
print("Kiem dinh cheo 5 phan tren tap hoc:")
diem = {}
for k in range(1, 52, 2):
    diem[k] = cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              X_tr, y_tr, cv=5).mean()
k_tot = max(diem, key=diem.get)
for k in [1, 5, 9, 13, 17, 25, 51]:
    dau = "<- tot nhat" if k == k_tot else ""
    print(f"k = {k:2d}    diem trung binh {diem[k]:.4f} {dau}")
print()

# Chọn xong mới mởtập kiểm tra, và chỉmởmột lần
mh = KNeighborsClassifier(n_neighbors=k_tot).fit(X_tr, y_tr)
print(f"k tot nhat = {k_tot}")
acc = accuracy_score(y_te, mh.predict(X_te))
print(f"Accuracy tren tap kiem tra = {acc:.4f}")


Accuracy tren tap hoc va tap kiem tra theo k
  k      tap hoc    tap kiem tra
  1      1.0000        0.8000
  5      0.8756        0.9067
 13      0.8844        0.9200
 31      0.8667        0.8933
101      0.8311        0.8533

Kiem dinh cheo 5 phan tren tap hoc:
k =  1    diem trung binh 0.7778 
k =  5    diem trung binh 0.8444 
k =  9    diem trung binh 0.8444 
k = 13    diem trung binh 0.8667 <- tot nhat
k = 17    diem trung binh 0.8533 
k = 25    diem trung binh 0.8622 
k = 51    diem trung binh 0.8533 

k tot nhat = 13
Accuracy tren tap kiem tra = 0.9200


## 4 - So sánh KNN với Logistic và Naive Bayes

In [4]:
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

cac_mo_hinh = [
    ("Hoi quy logistic", LogisticRegression()),
    ("Naive Bayes", GaussianNB()),
    ("KNN, k = 13", KNeighborsClassifier(n_neighbors=13)),
]

print("Accuracy tren tap kiem tra, bo hai vang trang")
for ten, mh in cac_mo_hinh:
    mh.fit(X_tr, y_tr)
    print(f"{ten}: {accuracy_score(y_te, mh.predict(X_te)):.4f}")


Accuracy tren tap kiem tra, bo hai vang trang
Hoi quy logistic: 0.8533
Naive Bayes: 0.8533
KNN, k = 13: 0.9200


## 5 - KNN cho bài toán hồi quy

In [5]:
# -*- coding: utf-8 -*-

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv("data/gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]

# Cùng cách chia với bài 1 đểso sánh công bằng
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42)

knn = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)
duong = LinearRegression().fit(X_tr, y_tr)

print("MAE tren tap kiem tra (ty dong)")
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong.predict(X_te))
print(f"KNN, k = 5: {mae_knn:.4f}")
print(f"Duong thang : {mae_duong:.4f}")
print()

can_moi = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("Du doan cho bon can moi (ty dong)")
print("dien tich      KNN    duong thang")
for dt, a, b in zip(can_moi["dien_tich"], knn.predict(can_moi),
                    duong.predict(can_moi)):
    print(f"{dt:6.0f} m2      {a:6.2f}     {b:6.2f}")
print()

# Năm láng giềng của căn 250 m2 chính là những căn rộng nhất đã thấy
_, chi_so = knn.kneighbors(pd.DataFrame({"dien_tich": [250.0]}))
lg = X_tr.iloc[chi_so[0]]["dien_tich"].tolist()
print("Nam lang gieng cua can 250 m2:", ", ".join(f"{v:.1f}" for v in lg))


MAE tren tap kiem tra (ty dong)
KNN, k = 5: 0.4030
Duong thang : 0.2578

Du doan cho bon can moi (ty dong)
dien tich      KNN    duong thang
    60 m2        4.95       5.12
   100 m2        8.32       8.24
   150 m2        9.32      12.13
   250 m2        9.32      19.91

Nam lang gieng cua can 250 m2: 117.5, 115.9, 111.7, 111.2, 104.2


# Bài tập

## Bài tập 1 - Tự viết KNN không dùng thư viện

In [6]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv('data/hai_vang_trang.csv')
X_tr, X_te, y_tr, y_te = train_test_split(
    df[['x1', 'x2']].to_numpy(), df['nhan'].to_numpy(),
    test_size=0.25, random_state=17, stratify=df['nhan']
)

def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    vi_tri = np.argsort(khoang_cach)[:k]
    nhan_gan = y_hoc[vi_tri].astype(int)
    return int(np.bincount(nhan_gan).argmax())

y_tu_viet = np.array([knn_du_doan(X_tr, y_tr, x, 13) for x in X_te])
mo_hinh = KNeighborsClassifier(n_neighbors=13).fit(X_tr, y_tr)
y_sklearn = mo_hinh.predict(X_te)

print(f'Accuracy tu viet : {accuracy_score(y_te, y_tu_viet):.4f}')
print(f'Accuracy sklearn: {accuracy_score(y_te, y_sklearn):.4f}')
print('So du doan khac nhau:', (y_tu_viet != y_sklearn).sum())


Accuracy tu viet : 0.9200
Accuracy sklearn: 0.9200
So du doan khac nhau: 0


- Tính khoảng cách từ điểm mới tới các điểm học, lấy 13 điểm gần nhất rồi chọn nhãn nhiều phiếu
- Cùng cách đo khoảng cách và cùng dữ liệu thì hai cách dự đoán cần cho kết quả giống nhau

## Bài tập 2 - Khoảng cách Manhattan

In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv('data/hai_vang_trang.csv')
X_tr, X_te, y_tr, y_te = train_test_split(
    df[['x1', 'x2']], df['nhan'],
    test_size=0.25, random_state=17, stratify=df['nhan']
)

for metric in ['euclidean', 'manhattan']:
    diem = {}
    for k in range(1, 52, 2):
        mo_hinh = KNeighborsClassifier(n_neighbors=k, metric=metric)
        diem[k] = cross_val_score(mo_hinh, X_tr, y_tr, cv=5).mean()
    k_tot = max(diem, key=diem.get)
    mo_hinh = KNeighborsClassifier(n_neighbors=k_tot, metric=metric).fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mo_hinh.predict(X_te))
    print(f'{metric:10s} | k tot nhat = {k_tot:2d} | CV = {diem[k_tot]:.4f} | Test = {acc:.4f}')


euclidean  | k tot nhat = 13 | CV = 0.8667 | Test = 0.9200
manhattan  | k tot nhat = 27 | CV = 0.8667 | Test = 0.9067


- Euclidean đo khoảng cách theo đường thẳng, Manhattan cộng khoảng cách theo từng trục
- So sánh Accuracy của hai cách để xem việc đổi khoảng cách ảnh hưởng nhiều hay ít

## Bài tập 3 - Đổi đơn vị đo

In [8]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('data/sinh_vien_mo_rong.csv')
cot = ['gio_on', 'diem_giua_ky', 'chi_tieu']
X_goc = df[cot].copy()
X_doi = X_goc.copy()
X_doi['chi_tieu'] /= 1000

y = df['qua_mon']
idx_tr, idx_te = train_test_split(
    df.index, test_size=0.25, random_state=17, stratify=y
)

def cham(X, chuan_hoa=False):
    if chuan_hoa:
        mo_hinh = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
    else:
        mo_hinh = KNeighborsClassifier(n_neighbors=7)
    mo_hinh.fit(X.loc[idx_tr], y.loc[idx_tr])
    return accuracy_score(y.loc[idx_te], mo_hinh.predict(X.loc[idx_te]))

print(f'Goc khong chuan hoa : {cham(X_goc):.4f}')
print(f'Doi don vi, khong chuan hoa: {cham(X_doi):.4f}')
print(f'Goc co chuan hoa    : {cham(X_goc, True):.4f}')
print(f'Doi don vi, co chuan hoa   : {cham(X_doi, True):.4f}')


Goc khong chuan hoa : 0.5333
Doi don vi, khong chuan hoa: 0.8667
Goc co chuan hoa    : 0.8667
Doi don vi, co chuan hoa   : 0.8667


- Đổi `chi_tieu` từ nghìn sang triệu làm cột này có thang đo nhỏ hơn, nên KNN không chuẩn hóa có thể đổi dự đoán
- Khi dùng StandardScaler, thay đổi đơn vị đo không làm thay đổi khoảng cách đã chuẩn hóa đáng kể

## Bài tập 4 - Láng giềng có trọng số

In [9]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

df = pd.read_csv('data/hai_vang_trang.csv')
X_tr, X_te, y_tr, y_te = train_test_split(
    df[['x1', 'x2']], df['nhan'],
    test_size=0.25, random_state=17, stratify=df['nhan']
)

print(' k      uniform    distance')
for k in [1, 13, 101]:
    ket_qua = []
    for weights in ['uniform', 'distance']:
        mo_hinh = KNeighborsClassifier(n_neighbors=k, weights=weights).fit(X_tr, y_tr)
        ket_qua.append(accuracy_score(y_te, mo_hinh.predict(X_te)))
    print(f'{k:3d}      {ket_qua[0]:.4f}      {ket_qua[1]:.4f}')


 k      uniform    distance
  1      0.8000      0.8000
 13      0.9200      0.9333
101      0.8533      0.9067


- Với `weights="distance"`, các điểm gần có tiếng nói lớn hơn khi dự đoán
- Với k lớn, cho điểm gần trọng số lớn có thể giúp mô hình ít bị ảnh hưởng bởi điểm ở xa

## Bài tập 5 - Chọn k cho bài toán hồi quy

In [10]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor

df = pd.read_csv('data/gia_nha.csv')
X_tr, X_te, y_tr, y_te = train_test_split(
    df[['dien_tich']], df['gia'], test_size=0.2, random_state=42
)

diem = {}
for k in range(1, 21):
    mo_hinh = KNeighborsRegressor(n_neighbors=k)
    diem[k] = -cross_val_score(
        mo_hinh, X_tr, y_tr, cv=5, scoring='neg_mean_absolute_error'
    ).mean()

k_tot = min(diem, key=diem.get)
knn = KNeighborsRegressor(n_neighbors=k_tot).fit(X_tr, y_tr)
duong_thang = LinearRegression().fit(X_tr, y_tr)
mae_knn = mean_absolute_error(y_te, knn.predict(X_te))
mae_duong = mean_absolute_error(y_te, duong_thang.predict(X_te))

print(f'k tot nhat = {k_tot} | CV MAE = {diem[k_tot]:.4f}')
print(f'Test MAE KNN    : {mae_knn:.4f}')
print(f'Test MAE duong thang: {mae_duong:.4f}')
print('KNN tot hon?', mae_knn < mae_duong)


k tot nhat = 2 | CV MAE = 0.3498
Test MAE KNN    : 0.4867
Test MAE duong thang: 0.2578
KNN tot hon? False


- Chọn k bằng 5-fold cross-validation trên tập học, không chọn theo tập kiểm tra
- MAE càng nhỏ thì mô hình dự đoán giá càng gần giá thật

## Bài tập 6 - Đo thời gian dự đoán

In [11]:
import time
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(0)
X_moi = rng.random((1000, 2))

print('So mau      fit (giay)     predict 1000 mau (giay)')
for n in [1000, 10000, 100000]:
    X = rng.random((n, 2))
    y = rng.integers(0, 2, size=n)
    mo_hinh = KNeighborsClassifier(n_neighbors=13)

    bat_dau = time.perf_counter()
    mo_hinh.fit(X, y)
    tg_fit = time.perf_counter() - bat_dau

    bat_dau = time.perf_counter()
    _ = mo_hinh.predict(X_moi)
    tg_predict = time.perf_counter() - bat_dau

    print(f'{n:6d}        {tg_fit:8.4f}               {tg_predict:8.4f}')


So mau      fit (giay)     predict 1000 mau (giay)
  1000          0.0022                 0.0804
 10000          0.0086                 0.0950
100000          0.1413                 0.1271


- KNN lưu dữ liệu lúc `fit`, còn lúc `predict` mới cần tìm các điểm gần nhất
- Dữ liệu học càng lớn, thời gian dự đoán thường tăng, vì vậy KNN được gọi là mô hình học lười